<a href="https://colab.research.google.com/github/leila-ali/IB2AD0_Data_Science_GenerativeAI/blob/main/My-work/7_02_Machine_Learning_Modelling_Student_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [ ]:
X = df.drop(columns='loan_status')
y = df['loan_status']

print(X.shape)
print(y.shape)

(50000, 15)
(50000,)


## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)


(40000, 15) (10000, 15)
(40000,) (10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)


(40000, 15) (10000, 15)


## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Logistic regression → scaled data
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Random forest → unscaled data
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# GBDT → unscaled data
gbdt = GradientBoostingClassifier(random_state=42)
gbdt.fit(X_train, y_train)

print("All three models trained")


All three models trained


## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def scores(model, X, y):
    pred = model.predict(X)
    return [
        accuracy_score(y, pred),
        precision_score(y, pred, average='macro'),
        recall_score(y, pred, average='macro'),
        f1_score(y, pred, average='macro')
    ]

models = {
    'Logistic Regression': (log_reg, X_train_scaled, X_test_scaled),
    'Random Forest': (rf, X_train, X_test),
    'GBDT': (gbdt, X_train, X_test)
}

rows = []
for name, (model, Xtr, Xte) in models.items():
    rows.append([name, 'Train'] + scores(model, Xtr, y_train))
    rows.append([name, 'Test'] + scores(model, Xte, y_test))

results = pd.DataFrame(rows, columns=['Model', 'Set', 'Accuracy', 'Precision', 'Recall', 'F1'])
results.round(3)


,Model,Set,Accuracy,Precision,Recall,F1
0,Logistic Regression,Train,0.839,0.838,0.836,0.837
1,Logistic Regression,Test,0.837,0.836,0.834,0.835
2,Random Forest,Train,1.000,1.000,1.000,1.000
3,Random Forest,Test,0.884,0.884,0.882,0.883
4,GBDT,Train,0.892,0.892,0.889,0.891
5,GBDT,Test,0.887,0.887,0.884,0.885


GBDT generalises best, with the highest test F1 (0.885) and almost no gap between training (0.891) and test performance. Random forest shows clear evidence of overfitting: it scores perfectly on the training data (1.000 on every metric) but drops to 0.883 F1 on the test set, meaning it has partly memorised the training data. Logistic regression does not overfit (0.837 vs 0.835) but performs worst overall, likely because a linear boundary is too simple to capture the patterns in the data.

## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?

Based on test results, I would choose GBDT: it has the best test performance and the smallest train–test gap.
Before using it for real lending decisions, I would want to:
	•	Check the confusion matrix. Missing a real default (false negative) costs the lender more than rejecting a good borrower, so recall on the default class matters more than overall accuracy.
	•	Use cross-validation, to make sure the results hold across different splits and not just this one.
	•	Check fairness, i.e. that the model doesn't treat groups unfairly (e.g. by age or gender proxies).
	•	Make it explainable. Lenders need to be able to justify why an applicant was rejected.
	•	Test on newer data, because borrower behaviour changes over time.